### imports

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import os
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from losses.final_loss import InverseDistanceBoundaryDiceLoss
from models.cbam_nnunet import nnUNet3D_CBAM
from utils.unets_helper_functions import (
    set_seed,
    save_checkpoint,
    final_PatchDataset_cbam,
    evaluate_full_volume_cbam,
    compute_per_class_dice
    
)

### data loading

In [3]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))


Train cases: 33
Val cases: 9


In [5]:
PATCH_SIZE = 96
PATCHES_PER_CASE = 12
FOREGROUND_PROB = 0.7
NUM_WORKERS = 2
train_dataset = final_PatchDataset_cbam(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = PATCHES_PER_CASE,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=True)

val_dataset = final_PatchDataset_cbam(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=NUM_WORKERS,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=NUM_WORKERS,    
    pin_memory=True
)

In [ ]:
model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24  
).to(device)

print("cbam nnU-Net style model Initialized")


cbam nnU-Net style model Initialized


In [7]:
EPOCHS = 100

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

weights = torch.tensor([0.05, 1, 1, 1.2, 1.8, 1.8, 1]).to(device)

loss_fn = InverseDistanceBoundaryDiceLoss(
    epsilon=1e-5,
    lambda_weight=0.4,
    class_weights = weights
)

scaler = torch.cuda.amp.GradScaler()

C:\Users\sajju\AppData\Local\Temp\ipykernel_8864\4270202072.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [11]:
def train_one_epoch_cbam(model, loader, optimizer, scaler, loss_fn, device):

    model.train()
    total_loss = 0

    for images, labels, dist_maps in tqdm(loader):

        images = images.to(device)
        labels = labels.long().to(device)
        dist_maps = dist_maps.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            outputs = model(images)

            # Deep supervision unpack
            out, ds2, ds3, ds4 = outputs


            loss_main = loss_fn(out, labels, dist_maps)
            loss_ds2  = loss_fn(ds2, labels, dist_maps)
            loss_ds3  = loss_fn(ds3, labels, dist_maps)
            loss_ds4  = loss_fn(ds4, labels, dist_maps)

            loss = (
                1.0 * loss_main +
                0.5 * loss_ds2 +
                0.25 * loss_ds3 +
                0.125 * loss_ds4
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

def validate_one_epoch_cbam(model, loader, loss_fn, device, num_classes=7):

    model.eval()
    total_loss = 0
    all_dices = []

    with torch.no_grad():
        for images, labels, dist_maps in loader:

            images = images.to(device)
            labels = labels.long().to(device)
            dist_maps = dist_maps.to(device)

            with torch.amp.autocast("cuda"):

                outputs = model(images)
                out = outputs[0]

                loss = loss_fn(out, labels, dist_maps)

            total_loss += loss.item()

            batch_dice = compute_per_class_dice(out, labels, num_classes)
            all_dices.append(batch_dice)

    mean_loss = total_loss / len(loader)

    all_dices = np.array(all_dices)
    mean_dices = np.mean(all_dices, axis=0)

    return mean_loss, mean_dices

#### weight - ([0.05, 1, 1, 1.2, 1.8, 1.8, 1])

In [ ]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "custom_loss_cbam")

os.makedirs(SAVE_DIR, exist_ok=True)

best_val_loss = float("inf")
best_dice = 0.0
best_parotid = 0.0
best_combined = 0.0

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    # ------------------ VALIDATE ------------------
    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    # Ignore background (class 0)
    mean_dice = val_dice[1:].mean()

    # Parotid (L=4, R=5)
    parotid_score = (val_dice[3] + val_dice[4]) / 2
    # 🔥 Combined metric (IMPORTANT)
    combined_score = 0.7 * mean_dice + 0.3 * parotid_score

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Parotid Dice (L,R): {val_dice[3]:.4f}, {val_dice[4]:.4f}")
    print(f"Combined Score: {combined_score:.4f}")

    # ------------------ SAVE BEST LOSS ------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "04_iw_best_loss_model.pth")
        )
        print("Saved Best Loss Model")

    # ------------------ SAVE BEST DICE ------------------
    if mean_dice > best_dice:
        best_dice = mean_dice
        save_checkpoint(
            model,
            optimizer,
            epoch,
            mean_dice,
            os.path.join(SAVE_DIR, "04_iw_best_dice_model.pth")
        )
        print("Saved Best Dice Model")

    # ------------------ SAVE BEST PAROTID ------------------
    if parotid_score > best_parotid:
        best_parotid = parotid_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            parotid_score,
            os.path.join(SAVE_DIR, "04_iw_best_parotid_model.pth")
        )
        print("Saved Best Parotid Model")

    # ------------------ SAVE BEST COMBINED ------------------
    if combined_score > best_combined:
        best_combined = combined_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            combined_score,
            os.path.join(SAVE_DIR, "04_iw_best_combined_model.pth")
        )
        print("Saved Best Combined Model")

    # ------------------ SAVE LAST MODEL ------------------
    save_checkpoint(
        model,
        optimizer,
        epoch,
        val_loss,
        os.path.join(SAVE_DIR, "04_iw_last_model.pth")
    )

    # ------------------ LR PRINT ------------------
    lr = optimizer.param_groups[0]['lr']
    print(f"LR: {lr:.6f}")

    # ------------------ SCHEDULER ------------------
    scheduler.step()

NameError: name 'train_one_epoch_cbam' is not defined

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24   # or try 32
).to(device)

model_path = "../experiments/final_cbam_fold0/best_combined_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


C:\Users\sajju\AppData\Local\Temp\ipykernel_8864\3396178766.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Model loaded for testing.


In [11]:
mean_dice, std_dice = evaluate_full_volume_cbam(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Volume: 272x357x357 | Patches: 245 | Stride: 48
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.9221246965498694), np.float64(0.860959124669328), np.float64(0.7917169977709801), np.float64(0.8491824442242013), np.float64(0.835006891282726), np.float64(0.9043543123721484)]
Volume: 243x383x383 | Patches: 245 | Stride: 48
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.9149963334407858), np.float64(0.7965223502269936), np.float64(0.7197753996845492), np.float64(0.8137167579757496), np.float64(0.8540613004362554), np.float64(0.8933584892781475)]
Volume: 264x333x333 | Patches: 180 | Stride: 48
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.8701675978068101), np.float64(0.7912959069800227), np.float64(0.7464767392840177), np.float64(0.7878436209560864), np.float64(0.793602437323989), np.float64(0.8811306441701358)]
Volume: 248x395x395 | Patche

In [13]:
model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24   # or try 32
).to(device)

print("cbam nnU-Net style model Initialized")


cbam nnU-Net style model Initialized


### revised weights - ([0.05, 1, 1.2, 1.5, 2, 2, 1])

In [15]:
EPOCHS = 40

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

weights = torch.tensor([0.05, 1, 1.2, 1.5, 2, 2, 1]).to(device)

loss_fn = InverseDistanceBoundaryDiceLoss(
    epsilon=1e-5,
    lambda_weight=0.4,
    class_weights = weights
)

scaler = torch.cuda.amp.GradScaler()

C:\Users\sajju\AppData\Local\Temp\ipykernel_8864\1239421176.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "custom_loss_cbam")

os.makedirs(SAVE_DIR, exist_ok=True)

best_val_loss = float("inf")
best_dice = 0.0
best_parotid = 0.0
best_combined = 0.0

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    # ------------------ VALIDATE ------------------
    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    # Ignore background (class 0)
    mean_dice = val_dice[1:].mean()

    # Parotid (L=4, R=5)
    parotid_score = (val_dice[3] + val_dice[4]) / 2
    # 🔥 Combined metric (IMPORTANT)
    combined_score = 0.7 * mean_dice + 0.3 * parotid_score

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Parotid Dice (L,R): {val_dice[3]:.4f}, {val_dice[4]:.4f}")
    print(f"Combined Score: {combined_score:.4f}")

    # ------------------ SAVE BEST LOSS ------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "04_rw_best_loss_model.pth")
        )
        print("Saved Best Loss Model")

    # ------------------ SAVE BEST DICE ------------------
    if mean_dice > best_dice:
        best_dice = mean_dice
        save_checkpoint(
            model,
            optimizer,
            epoch,
            mean_dice,
            os.path.join(SAVE_DIR, "04_rw_best_dice_model.pth")
        )
        print("Saved Best Dice Model")

    # ------------------ SAVE BEST PAROTID ------------------
    if parotid_score > best_parotid:
        best_parotid = parotid_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            parotid_score,
            os.path.join(SAVE_DIR, "04_rw_best_parotid_model.pth")
        )
        print("Saved Best Parotid Model")

    # ------------------ SAVE BEST COMBINED ------------------
    if combined_score > best_combined:
        best_combined = combined_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            combined_score,
            os.path.join(SAVE_DIR, "04_rw_best_combined_model.pth")
        )
        print("Saved Best Combined Model")

    # ------------------ SAVE LAST MODEL ------------------
    save_checkpoint(
        model,
        optimizer,
        epoch,
        val_loss,
        os.path.join(SAVE_DIR, "04_rw_last_model.pth")
    )

    # ------------------ LR PRINT ------------------
    lr = optimizer.param_groups[0]['lr']
    print(f"LR: {lr:.6f}")

    # ------------------ SCHEDULER ------------------
    scheduler.step()

100%|██████████| 198/198 [05:55<00:00,  1.80s/it]



Epoch 0
Train Loss: 0.7235
Val Loss: 0.3127
Per Class Dice: [4.58449692e-01 2.22222265e-01 1.11112232e-01 3.33333335e-01
 5.56831040e-07 4.14338910e-01]
Mean Dice: 0.2162
Parotid Dice (L,R): 0.3333, 0.0000
Combined Score: 0.2013
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 198/198 [05:50<00:00,  1.77s/it]



Epoch 1
Train Loss: 0.4732
Val Loss: 0.1931
Per Class Dice: [0.65689332 0.79878495 0.56670543 0.38501603 0.62877414 0.88347527]
Mean Dice: 0.6526
Parotid Dice (L,R): 0.3850, 0.6288
Combined Score: 0.6089
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 198/198 [06:09<00:00,  1.87s/it]


...


Epoch 37
Train Loss: 0.1831
Val Loss: 0.0602
Per Class Dice: [0.73979389 0.92487469 0.84022282 0.82727747 0.98290718 0.82549316]
Mean Dice: 0.8802
Parotid Dice (L,R): 0.8273, 0.9829
Combined Score: 0.8876
LR: 0.000003


100%|██████████| 198/198 [05:53<00:00,  1.79s/it]



Epoch 38
Train Loss: 0.1919
Val Loss: 0.0837
Per Class Dice: [0.82181909 0.85948243 0.88173397 0.80679364 0.89916029 0.89007382]
Mean Dice: 0.8674
Parotid Dice (L,R): 0.8068, 0.8992
Combined Score: 0.8631
LR: 0.000001


100%|██████████| 198/198 [05:57<00:00,  1.81s/it]



Epoch 39
Train Loss: 0.1810
Val Loss: 0.0657
Per Class Dice: [0.82569826 0.72200863 0.67997596 0.71733968 0.7580368  0.89089301]
Mean Dice: 0.7537
Parotid Dice (L,R): 0.7173, 0.7580
Combined Score: 0.7489
LR: 0.000000


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24   # or try 32
).to(device)

model_path = "../experiments/custom_loss_cbam/04_rw_best_combined_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


C:\Users\sajju\AppData\Local\Temp\ipykernel_8864\2982874843.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Model loaded for testing.


In [18]:
mean_dice, std_dice = evaluate_full_volume_cbam(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Volume: 272x357x357 | Patches: 245 | Stride: 48
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.927073467750758), np.float64(0.8250788644912628), np.float64(0.8397909536136404), np.float64(0.8424947147542339), np.float64(0.8215906849994171), np.float64(0.8787785708051714)]
Volume: 243x383x383 | Patches: 245 | Stride: 48
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.9235382309074847), np.float64(0.7611491567052371), np.float64(0.7296726509509457), np.float64(0.8274063236672776), np.float64(0.8328539470547434), np.float64(0.8970268930438825)]
Volume: 264x333x333 | Patches: 180 | Stride: 48
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.9099936261162758), np.float64(0.8315108579766588), np.float64(0.8270083787957806), np.float64(0.832150943522905), np.float64(0.8325785137481784), np.float64(0.8857138148867497)]
Volume: 248x395x395 | Patch

### spinal favored patching 

In [7]:
model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24   # or try 32
).to(device)

print("cbam nnU-Net style model Initialized")


cbam nnU-Net style model Initialized


In [8]:
EPOCHS = 40

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

weights = torch.tensor([0.05, 1, 1.3, 1.8, 2, 2, 1]).to(device)

loss_fn = InverseDistanceBoundaryDiceLoss(
    epsilon=1e-5,
    lambda_weight=0.4,
    class_weights = weights
)

scaler = torch.cuda.amp.GradScaler()

C:\Users\sajju\AppData\Local\Temp\ipykernel_15072\3779698161.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [12]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "custom_loss_cbam")

os.makedirs(SAVE_DIR, exist_ok=True)

best_val_loss = float("inf")
best_dice = 0.0
best_parotid = 0.0
best_combined = 0.0

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    # ------------------ VALIDATE ------------------
    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    # Ignore background (class 0)
    mean_dice = val_dice[1:].mean()

    # Parotid (L=4, R=5)
    parotid_score = (val_dice[3] + val_dice[4]) / 2
    # 🔥 Combined metric (IMPORTANT)
    combined_score = 0.7 * mean_dice + 0.3 * parotid_score

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Parotid Dice (L,R): {val_dice[3]:.4f}, {val_dice[4]:.4f}")
    print(f"Combined Score: {combined_score:.4f}")

    # ------------------ SAVE BEST LOSS ------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "04_sc_best_loss_model.pth")
        )
        print("Saved Best Loss Model")

    # ------------------ SAVE BEST DICE ------------------
    if mean_dice > best_dice:
        best_dice = mean_dice
        save_checkpoint(
            model,
            optimizer,
            epoch,
            mean_dice,
            os.path.join(SAVE_DIR, "04_sc_best_dice_model.pth")
        )
        print("Saved Best Dice Model")

    # ------------------ SAVE BEST PAROTID ------------------
    if parotid_score > best_parotid:
        best_parotid = parotid_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            parotid_score,
            os.path.join(SAVE_DIR, "04_sc_best_parotid_model.pth")
        )
        print("Saved Best Parotid Model")

    # ------------------ SAVE BEST COMBINED ------------------
    if combined_score > best_combined:
        best_combined = combined_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            combined_score,
            os.path.join(SAVE_DIR, "04_sc_best_combined_model.pth")
        )
        print("Saved Best Combined Model")

    # ------------------ SAVE LAST MODEL ------------------
    save_checkpoint(
        model,
        optimizer,
        epoch,
        val_loss,
        os.path.join(SAVE_DIR, "04_sc_last_model.pth")
    )

    # ------------------ LR PRINT ------------------
    lr = optimizer.param_groups[0]['lr']
    print(f"LR: {lr:.6f}")

    # ------------------ SCHEDULER ------------------
    scheduler.step()

100%|██████████| 198/198 [05:45<00:00,  1.75s/it]



Epoch 0
Train Loss: 0.6713
Val Loss: 0.2481
Per Class Dice: [0.55383511 0.32923467 0.1764637  0.20525607 0.22222446 0.61146775]
Mean Dice: 0.3089
Parotid Dice (L,R): 0.2053, 0.2222
Combined Score: 0.2804
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 198/198 [05:53<00:00,  1.79s/it]



Epoch 1
Train Loss: 0.3525
Val Loss: 0.0729
Per Class Dice: [0.64266226 0.55695556 0.73442239 0.40659277 0.52638326 0.76641407]
Mean Dice: 0.5982
Parotid Dice (L,R): 0.4066, 0.5264
Combined Score: 0.5587
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 198/198 [05:58<00:00,  1.81s/it]


...


Epoch 37
Train Loss: 0.1764
Val Loss: 0.0751
Per Class Dice: [0.81260267 0.75480664 0.8496281  0.64593192 0.79731268 0.89692265]
Mean Dice: 0.7889
Parotid Dice (L,R): 0.6459, 0.7973
Combined Score: 0.7687
LR: 0.000003


100%|██████████| 198/198 [05:44<00:00,  1.74s/it]



Epoch 38
Train Loss: 0.1679
Val Loss: 0.0589
Per Class Dice: [0.95848953 0.69306105 0.8858419  0.83236391 0.86825248 0.90110632]
Mean Dice: 0.8361
Parotid Dice (L,R): 0.8324, 0.8683
Combined Score: 0.8404
LR: 0.000001


100%|██████████| 198/198 [05:51<00:00,  1.78s/it]



Epoch 39
Train Loss: 0.1709
Val Loss: 0.0750
Per Class Dice: [0.82814556 0.76275625 0.79127906 0.74265079 0.68364233 0.76492885]
Mean Dice: 0.7491
Parotid Dice (L,R): 0.7427, 0.6836
Combined Score: 0.7383
LR: 0.000000


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24
).to(device)

model_path = "../experiments/custom_loss_cbam/04_sc_best_combined_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


C:\Users\sajju\AppData\Local\Temp\ipykernel_15072\2547328426.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Model loaded for testing.


In [14]:
mean_dice, std_dice = evaluate_full_volume_cbam(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Volume: 272x357x357 | Patches: 245 | Stride: 48
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.9338378264927643), np.float64(0.8561212593423077), np.float64(0.8182610214004861), np.float64(0.8376716541938187), np.float64(0.8402318724293381), np.float64(0.9089318291989776)]
Volume: 243x383x383 | Patches: 245 | Stride: 48
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.9145614035337539), np.float64(0.67306865438364), np.float64(0.7346938780098687), np.float64(0.8214687852987184), np.float64(0.8317375208529698), np.float64(0.898038055133256)]
Volume: 264x333x333 | Patches: 180 | Stride: 48
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.9018812009077821), np.float64(0.8415841585328918), np.float64(0.8136328430462698), np.float64(0.8267786125889095), np.float64(0.8149386846512672), np.float64(0.902296410037844)]
Volume: 248x395x395 | Patches

### Results

### Model Evaluation Comparison

### lambda = 0.4

| Model & Configuration | Class 1 | Class 2 | Class 3 | Class 4 (Parotid L) | Class 5 (Parotid R) | Class 6 | Mean Dice |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Model 1 (Initial Weights)**<br>`[0.05, 1, 1, 1.2, 1.8, 1.8, 1]` | 0.8960 | 0.8017 | 0.7107 | 0.8102 | 0.7981 | 0.8434 | 0.8100 |
| **Model 2 (Revised Weights)**<br>`[0.05, 1, 1.2, 1.5, 2, 2, 1]` | **0.9132** | **0.8017** | **0.7583** | 0.8170 | 0.7837 | 0.8353 | **0.8182** |
| **Model 3 (Spinal Favored Patching)**<br>`[0.05, 1, 1.3, 1.8, 2, 2, 1]` | 0.9119 | 0.7586 | 0.7461 | **0.8180** | **0.8033** | **0.8556** | 0.8156 |

**Observations**:
* **Model 2 (Revised Weights)** achieved the highest overall **Mean Dice (0.8182)** and performed best on Classes 1, 2, and 3.
* **Model 3 (Spinal Favored Patching)** sacrificed some performance on Class 2, but achieved the best results for the Parotid glands (Classes 4 & 5) and Class 6.


### Model Evaluation Comparison (Lambda = 0.6)

| Model & Configuration | Class 1 | Class 2 | Class 3 | Class 4 (Parotid L) | Class 5 (Parotid R) | Class 6 | Mean Dice |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Model 1**<br>`[0.05, 1, 1, 1.2, 1.8, 1.8, 1]` | 0.9047 ± 0.0235 | **0.8100 ± 0.0349** | 0.7394 ± 0.0585 | 0.7768 ± 0.0279 | 0.7894 ± 0.0770 | 0.8007 ± 0.0609 | 0.8035 |
| **Model 2**<br>`[0.05, 1, 1.2, 1.5, 2, 2, 1]` | **0.9142 ± 0.0266** | 0.8083 ± 0.0324 | **0.7553 ± 0.0526** | **0.8094 ± 0.0298** | **0.7975 ± 0.0571** | **0.8559 ± 0.0362** | **0.8234** |

**Observations (λ=0.6)**:
* **Model 2** significantly outperforms Model 1 across almost all classes, achieving a higher overall Mean Dice (0.8234).
* The Parotid glands (Classes 4 & 5) and Class 6 show noticeable improvements in Model 2 upon revising the class weights.
